# Lesson 02 — Makemore bigram language model

This notebook follows Karpathy's makemore lesson from the smallest useful language model: a **bigram** model.

The raw dataset lives at `data/raw/names.txt`. Each line is one name, and each name is treated as a sequence of characters.

The first goal is to learn counts like: given the current character, which next character usually follows? Later we will turn those counts into probabilities, sample new names, and compare this counting model with a tiny neural network.

Notebook path note: depending on how Jupyter starts the kernel, relative paths may be resolved from either the repository root or this notebook folder. The loading cell below handles both cases.


## 1. Load and inspect the names dataset

`words` will be a Python list of strings, where each string is one name such as `"emma"`.

The final line is a sanity check: it shows the first 10 names, the number of names, the shortest name length, and the longest name length. For the original makemore dataset, expect 32,033 names with lengths from 2 to 15 characters.


In [ ]:
from pathlib import Path

candidate_paths = [
    Path("data/raw/names.txt"),          # kernel launched from repo root
    Path("../../../data/raw/names.txt"), # kernel launched from this notebook folder
]
names_path = next(path for path in candidate_paths if path.exists())
words = names_path.read_text(encoding="utf-8").splitlines()

words[:10], len(words), min(len(w) for w in words), max(len(w) for w in words)


## 2. Count bigrams

A **bigram** is a pair of neighboring tokens. Here the tokens are characters, plus two special boundary tokens: `<S>` for the start of a name and `</S>` for the end of a name.

The dictionary `b` maps each pair `(previous_token, next_token)` to the number of times that pair appears in the dataset. For example, `('e', 'm')` counts how often `m` follows `e`.

These counts are the raw material for the first model: later, each row of counts becomes a probability distribution over the next character.


In [ ]:
b = {}
for w in words:
    chs = ["<S>"] + list(w) + ["</S>"]
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

b